# Binary Classification on a Healthcare Dataset
## Comparing Decision Tree, Naive Bayes and K-Nearest Neighbors for Cardiovascular Disease Prediction

**Course:** CSE 3208
**Submitted by:** Md Tayeb Ibne Sayed (Roll: 20230104027, Section A)
**Dataset:** Cardiovascular Disease Dataset (`cardiovascular_disease_dataset.csv`) — a hospital-collected clinical dataset of 1,000 patients (Mendeley Data, Mishra et al.)
**Target variable:** `target` — whether the patient has cardiovascular/heart disease (1 = yes, 0 = no)

This notebook builds and compares three classifiers — Decision Tree, Naive Bayes, and K-Nearest Neighbors — to predict the presence of cardiovascular disease from clinical and diagnostic measurements.

## Section 2: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# numpy is used later to mark implausible zero-cholesterol readings as missing
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
)

RANDOM_STATE = 42

## Section 3: Load Dataset

The dataset is the **Cardiovascular Disease Dataset**, a hospital-collected clinical dataset of
1,000 patients published on Mendeley Data. It records demographic and clinical/diagnostic
measurements (chest pain type, blood pressure, cholesterol, ECG results, exercise test results,
etc.) along with whether the patient has cardiovascular disease.

In [ ]:
import os
import urllib.request

DATA_FILE = "cardiovascular_disease_dataset.csv"
DATA_URL = ("https://raw.githubusercontent.com/Tayebbb/"
            "cardiovascular-disease-classification/master/cardiovascular_disease_dataset.csv")

# In Google Colab the runtime starts empty, so the CSV is not there yet.
# If it is not found locally, download it once from the public GitHub repo.
if not os.path.exists(DATA_FILE):
    urllib.request.urlretrieve(DATA_URL, DATA_FILE)

df = pd.read_csv(DATA_FILE)
df.head()

## Section 4: Initial Dataset Inspection

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df["target"].value_counts()

## Section 5: Data Preprocessing

**Irrelevant feature:** `patientid` is just a record identifier with no predictive value, so it
is dropped.

**Missing values:** `df.isnull().sum()` shows no `NaN` values, but `serumcholestrol` contains 53
rows recorded as **0 mg/dL**, which is not physiologically possible for a living patient — these
are missing values encoded as 0 rather than true `NaN`. Dropping these rows would risk falling
below the assignment's 1,000-sample minimum (leaving only 947), so they are instead marked as
missing here and imputed later (Section 6) using only the **training set's** median, to avoid
leaking any test-set information into preprocessing.

In [ ]:
df.isnull().sum()

In [ ]:
# Drop irrelevant features: 'patientid' is just a record ID with no predictive value.
# Leaving it would let the model "cheat" by memorizing patient IDs instead of learning real patterns.
df = df.drop(columns=["patientid"])

# Detect biologically-impossible values in 'serumcholestrol'.
# 53 rows are recorded as 0 mg/dL, which is not physiologically possible.
# These are missing values encoded as 0, not genuine readings.
# We mark them as NaN here; the replacement value (median) is computed later from
# the training set ONLY (after the split), so no test-set information leaks into preprocessing.
zero_chol_count = (df["serumcholestrol"] == 0).sum()
print("Rows with serumcholestrol == 0 (treated as missing):", zero_chol_count)
df["serumcholestrol"] = df["serumcholestrol"].replace(0, np.nan)

print("Shape after dropping patientid:", df.shape)
print("Target class balance:")
df["target"].value_counts()

In [ ]:
# Separate features and target
X = df.drop(columns=["target"])
y = df["target"]
X.head()

## Section 6: Train/Test Split

The data is split 80/20 with a fixed `random_state` for reproducibility, and stratified on the
target so both sets keep the same ~58%/42% class ratio.

In [ ]:
# Split the data 80/20 to avoid overfitting.
# - Training set (80%): used to teach the model
# - Test set (20%): held out, used ONLY at the end for honest evaluation
# - random_state=42: makes the split reproducible (same split every run)
# - stratify=y: ensures both sets have the same class ratio (~58/42), not just by random chance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Training set:", X_train.shape, " Testing set:", X_test.shape)

In [ ]:
# Impute missing 'serumcholestrol' values using TRAINING DATA ONLY.
# Computing the median from the full dataset (before split) would leak test-set info
# into preprocessing. Instead, we:
# 1. Compute median ONLY from X_train's valid readings
# 2. Apply that same median to BOTH X_train and X_test
# This keeps the test set clean and unbiased for final evaluation.
median_chol = X_train["serumcholestrol"].median()
X_train = X_train.copy()
X_test = X_test.copy()
X_train["serumcholestrol"] = X_train["serumcholestrol"].fillna(median_chol)
X_test["serumcholestrol"] = X_test["serumcholestrol"].fillna(median_chol)
print("Imputed missing serumcholestrol with training-set median =", median_chol)

## Section 7: Feature Scaling

KNN classifies points based on distance, so features measured on larger numeric scales (e.g.
`restingBP`, `serumcholestrol`) would dominate the distance calculation unless every feature is
put on a comparable scale. `StandardScaler` is **fit only on the training data**, then applied
to the test data, to avoid leaking test-set information into preprocessing.

Decision Tree and Naive Bayes do not require scaling (Decision Tree splits on raw thresholds;
GaussianNB models each feature's own distribution), but using the same scaled data for Naive
Bayes does not change its predictions mathematically, so scaled data is used for NB and KNN for
convenience, and raw data for the Decision Tree.

In [ ]:
# Scale features for KNN and Naive Bayes (not needed for Decision Tree).
# KNN uses distance between points: features on larger scales (e.g., cholesterol in hundreds)
# would dominate tiny scales (binary flags), breaking the distance calculation.
# StandardScaler: (x - mean) / std, so all features are on the same scale.
# FIT on training data ONLY, then apply to test data, to prevent test-set leakage.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # Learn mean/std from training, scale it
X_test_scaled = scaler.transform(X_test)        # Apply SAME scaling to test (no new learning)

## Section 8-10: Model Training

Three classifiers are initialized using Scikit-learn: Decision Tree, Naive Bayes (`GaussianNB`),
and K-Nearest Neighbors. Their key hyperparameters are chosen in Section 11 using cross-validation
on the training data only, then the final models below are trained with those chosen values.

## Section 11: Hyperparameter Experiment (Training Data Only)

A small, understandable experiment is used to choose hyperparameters, using 5-fold
cross-validation **on the training set only** (the test set is never touched during this step).
F1-score is used as the selection metric because the target is imbalanced (~85% negative /
15% positive), so accuracy alone could hide poor performance on the minority (CHD) class.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print("Decision Tree: max_depth experiment")
dt_depths = [3, 4, 5, 6, 8, 10, None]
dt_cv_scores = []
best_dt_depth, best_dt_score = None, -1
for depth in dt_depths:
    clf = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE)
    scores = cross_val_score(clf, X_train, y_train, cv=cv, scoring="f1")
    dt_cv_scores.append(scores.mean())
    print(f"max_depth={depth}: mean CV F1 = {scores.mean():.4f}")
    if scores.mean() > best_dt_score:
        best_dt_score, best_dt_depth = scores.mean(), depth
print("Selected max_depth:", best_dt_depth)

In [ ]:
depth_labels = [str(d) if d is not None else "None" for d in dt_depths]
plt.figure(figsize=(6, 4))
plt.plot(depth_labels, dt_cv_scores, marker="o")
plt.xlabel("max_depth")
plt.ylabel("Mean CV F1-score")
plt.title("Decision Tree: CV F1-score vs. max_depth")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
print("KNN: n_neighbors experiment")
knn_ks = [3, 5, 7, 9, 11, 15, 21]
knn_cv_scores = []
best_k, best_k_score = None, -1
for k in knn_ks:
    clf = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(clf, X_train_scaled, y_train, cv=cv, scoring="f1")
    knn_cv_scores.append(scores.mean())
    print(f"k={k}: mean CV F1 = {scores.mean():.4f}")
    if scores.mean() > best_k_score:
        best_k_score, best_k = scores.mean(), k
print("Selected n_neighbors:", best_k)

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(knn_ks, knn_cv_scores, marker="o")
plt.xlabel("k (n_neighbors)")
plt.ylabel("Mean CV F1-score")
plt.title("KNN: CV F1-score vs. Number of Neighbors (k)")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
print("Naive Bayes: var_smoothing experiment")
best_vs, best_vs_score = None, -1
for vs in [1e-9, 1e-8, 1e-7, 1e-6, 1e-5]:
    clf = GaussianNB(var_smoothing=vs)
    scores = cross_val_score(clf, X_train_scaled, y_train, cv=cv, scoring="f1")
    print(f"var_smoothing={vs}: mean CV F1 = {scores.mean():.4f}")
    if scores.mean() > best_vs_score:
        best_vs_score, best_vs = scores.mean(), vs
print("Selected var_smoothing:", best_vs)

## Section 12: Final Model Training and Test-Set Evaluation

The three models are now trained on the full training set using the hyperparameters selected
above, and evaluated **once** on the held-out test set.

In [ ]:
dt_model = DecisionTreeClassifier(max_depth=best_dt_depth, random_state=RANDOM_STATE)
dt_model.fit(X_train, y_train)

nb_model = GaussianNB(var_smoothing=best_vs)
nb_model.fit(X_train_scaled, y_train)

knn_model = KNeighborsClassifier(n_neighbors=best_k)
knn_model.fit(X_train_scaled, y_train)

dt_pred = dt_model.predict(X_test)
nb_pred = nb_model.predict(X_test_scaled)
knn_pred = knn_model.predict(X_test_scaled)

In [ ]:
def evaluate(name, y_true, y_pred):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-score": f1_score(y_true, y_pred, zero_division=0),
    }

results = pd.DataFrame([
    evaluate("Decision Tree", y_test, dt_pred),
    evaluate("Naive Bayes", y_test, nb_pred),
    evaluate("KNN", y_test, knn_pred),
])
results

## Section 13: Model Comparison

In [ ]:
results.set_index("Model")[["Accuracy", "Precision", "Recall", "F1-score"]].plot(
    kind="bar", figsize=(7, 4)
)
plt.ylabel("Score")
plt.title("Model Comparison on Test Set")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Section 14: Additional Visualization

In [ ]:
df["target"].value_counts().sort_index().plot(kind="bar", color=["#4C72B0", "#C44E52"], figsize=(5,4))
plt.xticks([0, 1], ["No disease (0)", "Disease (1)"], rotation=0)
plt.ylabel("Number of patients")
plt.title("Target Class Distribution")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (name, pred) in zip(
    axes, [("Decision Tree", dt_pred), ("Naive Bayes", nb_pred), ("KNN", knn_pred)]
):
    cm = confusion_matrix(y_test, pred)
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="black")
plt.tight_layout()
plt.show()

## Section 15: Conclusion

- **Decision Tree** achieved the best overall test performance: 97.5% accuracy and an F1-score
  of 0.978, with `max_depth=6` selected by cross-validation. Features such as `chestpain`,
  `noofmajorvessels`, and `exerciseangia` create fairly clean, rule-like splits between the two
  classes, which suits a tree-based model well.
- **Naive Bayes** achieved 94.0% accuracy and an F1-score of 0.949. Even though its
  feature-independence assumption is not exactly true here, the classes are well separated
  enough that the Gaussian approximation still works well.
- **KNN** achieved 93.5% accuracy and an F1-score of 0.944, with `n_neighbors=21` selected by
  cross-validation — a relatively large K, suggesting the two classes form broad, well-separated
  regions in the scaled feature space rather than requiring very local decision boundaries.
- All three models perform strongly and consistently (93-98% accuracy, F1 all above 0.94)
  because this dataset's clinical features (especially chest pain type, number of major
  vessels, exercise-induced angina, and ST-segment slope) are strongly associated with the
  presence of cardiovascular disease, and the classes are reasonably balanced (58%/42%).
- No attempt was made to artificially inflate accuracy (e.g. by rebalancing the test set or
  tuning on the test set); all reported numbers come directly from the single final evaluation
  on the untouched 20% test set, using hyperparameters chosen via training-only
  cross-validation.